# Stage 0 sweep on Colab (T4)

Runs a sweep grid (default **G-C**, the weight decay scouting grid: 84 runs) and writes a small results file you can hand over.

**Steps:** Runtime > Change runtime type > T4 GPU. Fill in `COMMIT` in the next cell. Then run the cells top to bottom. With `USE_DRIVE = True`, results are saved to your Google Drive after every batch of runs, so if Colab disconnects, run the cells again and it resumes where it stopped. With `USE_DRIVE = False` they stay on the Colab disk and are lost on a disconnect.

**Sharing one part between two Colabs:** for the second half of a 1,1 split, set `SPLIT = "1,1"`, `PART = 1`, `SUBSPLIT = "1,1"`, and `SUBPART = 0` on one Colab and `SUBPART = 1` on the other.

**Sharing a grid between machines:** use the same `SPLIT` on every machine and a different `PART` on each (for example `"7,3"` with part 0 here and `--part 1` on your PC). Leave `SPLIT = "1"` to run everything here.

The code is pinned to one commit so every machine runs identical code. `PRECISION = "fp32"` is the normal full-precision run and the one to compare against the Stage 0 baseline. `PRECISION = "fp16"` runs the big matrix products in half precision (faster on a T4 if it works); its results go to a separate folder (`<grid>-fp16`) and are only for comparing against the fp32 results.

In [ ]:
# ---- settings: edit these ----
REPO_URL  = "https://github.com/adi5022/Improving-Parameter-Efficiency-in-Transformer-Language-Models-Through-Superposition-Guided-Training.git"
COMMIT    = ""        # full commit hash with the G-C config (run `git rev-parse HEAD` on your PC after you commit and push)
GRID      = "G-D"     # G-D = Step B grid (seeds 3-9); G-C / G-A / G-B also work
PRECISION = "fp32"    # "fp32" = the normal runs; "fp16" = half-precision matrix products (results go to a separate folder, <grid>-fp16)
SPLIT     = "1"       # relative speeds of all machines, e.g. "7,3"; "1" = this machine does everything
PART      = 0         # which share this notebook runs (0-based)
SUBSPLIT  = ""        # to share ONE part between several Colabs, e.g. "1,1" (then SPLIT="1,1", PART=1 means the second half)
SUBPART   = 0         # which share of SUBSPLIT this notebook runs (0-based)
USE_DRIVE = False     # False = keep results on the Colab disk only (use this if the Drive permission popup fails); finished runs are then lost if Colab disconnects
DRIVE_DIR = "/content/drive/MyDrive/superposition_results" if USE_DRIVE else "/content/superposition_results"
# If the repository is private, add a GitHub token in Colab Secrets (key icon, left side) named GH_TOKEN.
assert COMMIT, "Fill in COMMIT first"
SUBARGS = f"--subsplit {SUBSPLIT} --subpart {SUBPART}" if SUBSPLIT else ""

In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type"
print("torch", torch.__version__)

In [ ]:
import os, subprocess
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass                      # no token: fine for a public repository
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", url, "/content/repo"], check=True)
%cd /content/repo
!git fetch --all -q
!git checkout -q {COMMIT}
!git rev-parse HEAD
!git status --short toy configs
!pip -q install numpy matplotlib

In [ ]:
# Keep results on Drive so a disconnect doesn't lose them (results/ is git-ignored, so the checkout stays clean).
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
!rm -rf results
os.symlink(DRIVE_DIR, "results")
!ls results

In [ ]:
# Quick speed check (about 30 seconds): trains 100 steps per width and estimates this machine's total time.
import time
from dataclasses import replace
from toy.batched import train_group
from toy.sweep import grid_configs, split_configs, make_groups

mine = split_configs(grid_configs(GRID), [float(w) for w in SPLIT.split(",")], PART)
if SUBSPLIT:
    mine = split_configs(mine, [float(w) for w in SUBSPLIT.split(",")], SUBPART)
groups = make_groups(mine, 24)
ms = {}
for g in groups:
    probe = [replace(c, steps=100, warmup_steps=20, test_batches=1, log_every=100) for c in g]
    train_group(probe[:2], device="cuda", half=(PRECISION == "fp16"))      # warm-up
    torch.cuda.synchronize(); t = time.time()
    train_group(probe, device="cuda", half=(PRECISION == "fp16")); torch.cuda.synchronize()
    ms[g[0].width] = (time.time() - t) / 100 * 1000
    print(f"width {g[0].width:>3}: {len(g)} runs together, {ms[g[0].width]:.1f} ms per step")
est = sum(ms[g[0].width] * g[0].steps / 1000 for g in groups) / 60
print(f"\n{len(mine)} runs here, estimated {est:.0f} min (rough; the 100-step test includes some setup cost)")

## Optional: is half precision (fp16) worth it? (about 3 minutes)

Trains the same 12 runs (width 63) twice, once in the normal float32 and once with the three big matrix products in half precision, then prints the speed ratio and how far the results differ. **Do this after the main run, not during it** (the GPU is shared). Half precision should only be used for real runs if the differences are small next to the seed-to-seed spread and the speedup is real.

In [ ]:
import time, statistics
from dataclasses import replace
from toy.batched import train_group
from toy.sweep import grid_configs, make_groups
g63 = [c for c in make_groups(grid_configs(GRID), 24) if c[0].width == 63][0]
cfgs = [replace(c, steps=5000, warmup_steps=500, test_batches=50, log_every=500) for c in g63]
out = {}
for half in (False, True):
    train_group(cfgs[:2], device="cuda", half=half)               # warm-up
    torch.cuda.synchronize(); t = time.time()
    out[half] = train_group(cfgs, device="cuda", half=half); torch.cuda.synchronize()
    out[half, "s"] = time.time() - t
print(f"float32: {out[False, 's']:.1f}s   fp16 products: {out[True, 's']:.1f}s   speedup {out[False, 's'] / out[True, 's']:.2f}x")
diffs, spread = [], []
for a, b in zip(out[False], out[True]):
    diffs.append(abs(b["test_loss"] - a["test_loss"]) / a["test_loss"])
by = {}
for a in out[False]:
    by.setdefault(a["config"]["gamma"], []).append(a["test_loss"])
spread = [statistics.pstdev(v) / statistics.mean(v) for v in by.values() if len(v) > 1]
print(f"fp16 vs float32, same run: median difference {100*statistics.median(diffs):.2f}%, worst {100*max(diffs):.2f}%")
print(f"seed-to-seed spread in float32 (relative s.d.): median {100*statistics.median(spread):.2f}%, worst {100*max(spread):.2f}%")
print("Looks safe to use" if max(diffs) < 0.5 * statistics.median(spread) and out[False, "s"] / out[True, "s"] > 1.3 else "Not clearly worth it: keep float32")

In [ ]:
# See what this notebook will run (trains nothing).
!python -m toy.sweep --grid {GRID} --split {SPLIT} --part {PART} {SUBARGS} --precision {PRECISION} --dry-run | tail -3

In [ ]:
# Run it. Safe to run again after a disconnect: finished runs are skipped.
!python -m toy.sweep --grid {GRID} --split {SPLIT} --part {PART} {SUBARGS} --precision {PRECISION}

## Results file

The next cell prints a table and makes **two files**:

- `<grid>[-fp16]_part<N>_small.zip`: just the run summaries and a report (a few KB). **This is the one to hand over.**
- `<grid>_part<N>_full.zip`: every run including the learned weights (tens of MB), only needed for the weight plots.

Save the small zip into your project (for example `results/colab/`) so it can be read.

In [ ]:
import os, json, shutil, collections, statistics, platform, datetime
SUFFIX = "-fp16" if PRECISION == "fp16" else ""
out = f"{DRIVE_DIR}/stage0/{GRID}{SUFFIX}"
rows = [json.loads(l) for l in open(f"{out}/summary.jsonl") if l.strip()]
by = collections.defaultdict(list)
for r in rows:
    by[(r["width"], r["gamma"])].append(r["test_loss"])
gammas = sorted({k[1] for k in by})
print("mean test error over seeds (relative s.d. in brackets)")
print("width | " + " | ".join(f"g={g:+.2f}" for g in gammas))
for w in sorted({k[0] for k in by}):
    cells = []
    for g in gammas:
        v = by.get((w, g))
        cells.append("   -   " if not v else f"{statistics.mean(v):.2e} ({100*statistics.pstdev(v)/statistics.mean(v):.0f}%)")
    print(f"{w:>5} | " + " | ".join(cells))

report = {
    "grid": GRID, "precision": PRECISION, "split": SPLIT, "part": PART, "subsplit": SUBSPLIT, "subpart": SUBPART, "commit": COMMIT,
    "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "python": platform.python_version(),
    "finished_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "runs_done": len(rows), "runs_not_ok": [r["run_id"] for r in rows if r["status"] != "ok"],
    "total_group_minutes": round(sum(r["train_seconds"] for r in rows) / 60, 1),
    "estimated_minutes_from_benchmark": round(est, 1) if "est" in globals() else None,
}
print("\n", json.dumps(report, indent=1))
tag = f"{GRID}{SUFFIX}_part{PART}" + (f"_sub{SUBPART}" if SUBSPLIT else "")
json.dump(report, open(f"{out}/report.json", "w"), indent=2)
os.makedirs("/content/small", exist_ok=True)
shutil.copy(f"{out}/summary.jsonl", "/content/small/summary.jsonl")
shutil.copy(f"{out}/report.json", "/content/small/report.json")
shutil.make_archive(f"/content/{tag}_small", "zip", "/content/small")
shutil.make_archive(f"/content/{tag}_full", "zip", out)
from google.colab import files
files.download(f"/content/{tag}_small.zip")
files.download(f"/content/{tag}_full.zip")